# AKT1 Hybrid Predictor -- Build, Freeze & External Validation (Merged)


## Setup: install dependencies (run this cell first)

In [ ]:
!pip install numpy pandas matplotlib scikit-learn requests joblib tqdm rdkit -q

---
# PART 1 -- Build & Freeze the Hybrid Predictor



# Redefine Custom Ensemble Classes Before Loading


In [ ]:
import joblib
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.base import clone, BaseEstimator, ClassifierMixin, RegressorMixin


class ProbaAverageEnsembleClassifier(BaseEstimator, ClassifierMixin):
    def __init__(self, estimator_a=None, estimator_b=None):
        self.estimator_a = estimator_a
        self.estimator_b = estimator_b

    def fit(self, X, y, sample_weight=None):
        if sample_weight is not None:
            self.estimator_a_ = clone(self.estimator_a).fit(X, y, sample_weight=sample_weight)
            self.estimator_b_ = clone(self.estimator_b).fit(X, y, sample_weight=sample_weight)
        else:
            self.estimator_a_ = clone(self.estimator_a).fit(X, y)
            self.estimator_b_ = clone(self.estimator_b).fit(X, y)
        self.classes_ = self.estimator_a_.classes_
        return self

    def predict_proba(self, X):
        return (self.estimator_a_.predict_proba(X) + self.estimator_b_.predict_proba(X)) / 2.0

    def predict(self, X):
        p = self.predict_proba(X)
        return self.classes_[np.argmax(p, axis=1)]


class WeightedBlendRegressor(BaseEstimator, RegressorMixin):
    def __init__(self, estimator_a=None, estimator_b=None, weight_a=0.5):
        self.estimator_a = estimator_a
        self.estimator_b = estimator_b
        self.weight_a = weight_a

    def fit(self, X, y):
        self.estimator_a_ = clone(self.estimator_a).fit(X, y)
        self.estimator_b_ = clone(self.estimator_b).fit(X, y)
        return self

    def predict(self, X):
        return self.weight_a * self.estimator_a_.predict(X) + (1 - self.weight_a) * self.estimator_b_.predict(X)


# Load, Combine, and Save

In [ ]:
classifier = joblib.load("classifier.joblib")
regressor = joblib.load("regressor.joblib")

hybrid = {
    "classifier": classifier,
    "regressor": regressor,
}
joblib.dump(hybrid, "hybrid_predictor.joblib")

print("Saved hybrid_predictor.joblib")
print(f"  classifier: {classifier['model_name']}")
print(f"  regressor:  {regressor['model_name']}")

try:
    from google.colab import files
    files.download("hybrid_predictor.joblib")
except Exception:
    print("Not running in Colab (or download failed) -- retrieve hybrid_predictor.joblib manually.")


# Freeze blend weight (w) and decision threshold -- development data only



In [ ]:
from sklearn.model_selection import GroupKFold
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import matthews_corrcoef, balanced_accuracy_score, f1_score
import datetime

features = joblib.load("features.pkl")
splits = joblib.load("splits.pkl")

clf_bundle_tmp = classifier   # from the cell above
reg_bundle_tmp = regressor

tune_idx = np.concatenate([splits["train_idx"], splits["val_idx"]])
tune_groups = np.asarray(features["scaffold"])[tune_idx]

tune_X_clf = pd.concat([splits["X_train"], splits["X_val"]], ignore_index=True)[clf_bundle_tmp["feature_columns"]]
tune_X_reg = pd.concat([splits["X_train"], splits["X_val"]], ignore_index=True)[reg_bundle_tmp["feature_columns"]]
tune_y_class = np.concatenate([splits["y_train_class"], splits["y_val_class"]])
tune_y_pic50 = np.concatenate([splits["y_train_pic50"], splits["y_val_pic50"]])

ACTIVE_CUTOFF = clf_bundle_tmp["active_pic50_cutoff"]
INACTIVE_CUTOFF = clf_bundle_tmp["inactive_pic50_cutoff"]
TIER_TO_LABEL = clf_bundle_tmp["tier_to_label"]
CLASS_LABELS_ORDERED = clf_bundle_tmp["class_labels_ordered"]
y_true_bin_tune = (tune_y_pic50 >= ACTIVE_CUTOFF).astype(int)

N_FOLDS = 5
gkf = GroupKFold(n_splits=N_FOLDS)

# --- Scaffold-grouped OOF classifier probabilities (clone of the exact final model/hyperparams) ---
oof_clf_proba = np.zeros((len(tune_y_class), len(CLASS_LABELS_ORDERED)))
print(f"Computing {N_FOLDS}-fold scaffold-grouped OOF classifier predictions on the development set...")
for fold_tr, fold_va in gkf.split(tune_X_clf, tune_y_class, groups=tune_groups):
    m = clone(clf_bundle_tmp["model"])
    y_fold = tune_y_class[fold_tr]
    try:
        sw = compute_sample_weight("balanced", y_fold)
        m.fit(tune_X_clf.iloc[fold_tr], y_fold, sample_weight=sw)
    except TypeError:
        m.fit(tune_X_clf.iloc[fold_tr], y_fold)
    oof_clf_proba[fold_va] = m.predict_proba(tune_X_clf.iloc[fold_va])

if clf_bundle_tmp["decision_bias"] is not None:
    oof_clf_proba = oof_clf_proba * np.exp(clf_bundle_tmp["decision_bias"])
    oof_clf_proba = oof_clf_proba / oof_clf_proba.sum(axis=1, keepdims=True)

# --- Scaffold-grouped OOF regressor predictions (clone of the exact final model/hyperparams) ---
oof_reg_pred = np.zeros(len(tune_y_pic50))
print(f"Computing {N_FOLDS}-fold scaffold-grouped OOF regressor predictions on the development set...")
for fold_tr, fold_va in gkf.split(tune_X_reg, tune_y_pic50, groups=tune_groups):
    m = clone(reg_bundle_tmp["model"])
    m.fit(tune_X_reg.iloc[fold_tr], tune_y_pic50[fold_tr])
    oof_reg_pred[fold_va] = m.predict(tune_X_reg.iloc[fold_va])

oof_reg_proba = np.zeros_like(oof_clf_proba)
for i, p in enumerate(oof_reg_pred):
    if p >= ACTIVE_CUTOFF:
        oof_reg_proba[i, TIER_TO_LABEL["Active"]] = 1.0
    elif p >= INACTIVE_CUTOFF:
        oof_reg_proba[i, TIER_TO_LABEL["Intermediate"]] = 1.0
    else:
        oof_reg_proba[i, TIER_TO_LABEL["Inactive"]] = 1.0

print(f"Development set (train+val): N={len(y_true_bin_tune)}, N_active={y_true_bin_tune.sum()}")



OPT_METRIC = "mcc"  # DECISION: <-- fill in your reasoning here, before running this cell
print(f"[PRE-REGISTERED {datetime.datetime.now().isoformat(timespec='minutes')}] "
      f"Optimizing w/threshold for '{OPT_METRIC}' on development-set OOF predictions only, "
      f"before touching external (BindingDB) data.")


def score_blend(w_, thr_):
    blended = w_ * oof_clf_proba + (1 - w_) * oof_reg_proba
    p_active = blended[:, TIER_TO_LABEL["Active"]]
    y_pred = (p_active >= thr_).astype(int)
    if len(np.unique(y_true_bin_tune)) < 2:
        return np.nan
    if OPT_METRIC == "mcc":
        return matthews_corrcoef(y_true_bin_tune, y_pred)
    elif OPT_METRIC == "balanced_acc":
        return balanced_accuracy_score(y_true_bin_tune, y_pred)
    elif OPT_METRIC == "f1":
        return f1_score(y_true_bin_tune, y_pred, zero_division=0)
    raise ValueError(f"Unknown OPT_METRIC: {OPT_METRIC}")

w_grid = np.round(np.arange(0.0, 1.0001, 0.05), 3)
thr_grid = np.round(np.arange(0.1, 0.9001, 0.025), 3)
sweep_rows = [{"w": w_, "threshold": thr_, OPT_METRIC: score_blend(w_, thr_)}
              for w_ in w_grid for thr_ in thr_grid]
sweep_df = pd.DataFrame(sweep_rows).dropna()
best_row = sweep_df.loc[sweep_df[OPT_METRIC].idxmax()]
BEST_W = float(best_row["w"])
BEST_THRESHOLD = float(best_row["threshold"])
print(f"\nBest (w, threshold) on development-set OOF predictions by {OPT_METRIC}: "
      f"w={BEST_W}, threshold={BEST_THRESHOLD}, {OPT_METRIC}={best_row[OPT_METRIC]:.4f}")


classifier["blend_weight"] = BEST_W
classifier["decision_threshold"] = BEST_THRESHOLD
hybrid = {"classifier": classifier, "regressor": regressor}
joblib.dump(hybrid, "hybrid_predictor.joblib")
print("\nFrozen pipeline saved to hybrid_predictor.joblib: preprocessing, feature columns, model "
      "hyperparameters, blend_weight, and decision_threshold are now locked. Evaluate on external "
      "(BindingDB) data exactly once from here -- no further tuning.")


# Sanity Check: Reload and Score a Few Molecules

Confirms the saved bundle round-trips correctly, and demonstrates the
intended way to use it: regenerate the exact same ECFP4 + RDKit + MACCS
hybrid features (+ 2D descriptors) for new SMILES, apply the same
leakage-safe filtering columns the models were trained on, then get both a
tier prediction and a continuous pIC50 from a single bundle.

In [ ]:
hybrid_reloaded = joblib.load("hybrid_predictor.joblib")


def featurize_smiles(smiles_list):
    """Regenerates the ECFP4 + RDKit-topological + MACCS hybrid fingerprint
    and 2D descriptors for new molecules, using the exact same functions as
    Notebook 1."""
    from rdkit import Chem, DataStructs
    from rdkit.Chem import Descriptors, rdFingerprintGenerator, MACCSkeys

    ecfp4_gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
    descriptor_funcs = [func for _, func in Descriptors._descList]
    descriptor_names = [name for name, _ in Descriptors._descList]

    def morgan_fp(mol):
        fp = ecfp4_gen.GetFingerprint(mol)
        arr = np.zeros((2048,), dtype=np.int8)
        DataStructs.ConvertToNumpyArray(fp, arr)
        return arr

    def rdkit_topo_fp(mol):
        fp = Chem.RDKFingerprint(mol, fpSize=2048)
        arr = np.zeros((2048,), dtype=np.int8)
        DataStructs.ConvertToNumpyArray(fp, arr)
        return arr

    def maccs_fp(mol):
        fp = MACCSkeys.GenMACCSKeys(mol)
        arr = np.zeros((167,), dtype=np.int8)
        DataStructs.ConvertToNumpyArray(fp, arr)
        return arr

    mols, valid_smiles = [], []
    for smi in smiles_list:
        mol = Chem.MolFromSmiles(smi)
        if mol is not None:
            mols.append(mol)
            valid_smiles.append(smi)

    hybrid_fps = np.array([np.concatenate([morgan_fp(m), rdkit_topo_fp(m), maccs_fp(m)]) for m in mols])
    desc_vals = []
    for m in mols:
        row = []
        for func in descriptor_funcs:
            try:
                row.append(func(m))
            except Exception:
                row.append(np.nan)
        desc_vals.append(row)

    fp_cols = [f"FP_{i}" for i in range(hybrid_fps.shape[1])]
    desc_cols = [f"DESC_{n}" for n in descriptor_names]
    X = pd.concat([
        pd.DataFrame(hybrid_fps, columns=fp_cols),
        pd.DataFrame(desc_vals, columns=desc_cols).replace([np.inf, -np.inf], np.nan),
    ], axis=1)
    return X, valid_smiles


def predict_akt1(smiles_list, bundle):
    clf_bundle, reg_bundle = bundle["classifier"], bundle["regressor"]
    X_raw, valid_smiles = featurize_smiles(smiles_list)


    X_filled_nan = X_raw.copy()
    X_filled_nan[reg_bundle["descriptor_medians"].index] = X_filled_nan[reg_bundle["descriptor_medians"].index].fillna(
        reg_bundle["descriptor_medians"]
    )

    # Create separate feature sets for classifier and regressor
    X_for_classifier = X_filled_nan.reindex(columns=clf_bundle["feature_columns"], fill_value=0)
    X_for_regressor = X_filled_nan.reindex(columns=reg_bundle["feature_columns"], fill_value=0)

    clf_pred = clf_bundle["model"].predict(X_for_classifier)
    reg_pred = reg_bundle["model"].predict(X_for_regressor)
    label_to_tier = clf_bundle["label_to_tier"]

    return pd.DataFrame({
        "smiles": valid_smiles,
        "classifier_tier": [label_to_tier[c] for c in clf_pred],
        "predicted_pIC50": reg_pred,
    })


example_smiles = ["CCO", "c1ccccc1O", "CC(=O)Oc1ccccc1C(=O)O"]  # ethanol, phenol, aspirin
predict_akt1(example_smiles, hybrid_reloaded)


# Additional Analysis: Cross-Model Consistency



In [ ]:
features = joblib.load("features.pkl")
splits = joblib.load("splits.pkl")

X_test_raw = splits["X_test"]
test_idx = splits["test_idx"]
y_test_class = splits["y_test_class"]

clf_model = hybrid_reloaded["classifier"]["model"]
reg_model = hybrid_reloaded["regressor"]["model"]
LABEL_TO_TIER = hybrid_reloaded["classifier"]["label_to_tier"]
ACTIVE_CUTOFF = hybrid_reloaded["classifier"]["active_pic50_cutoff"]
INACTIVE_CUTOFF = hybrid_reloaded["classifier"]["inactive_pic50_cutoff"]

# Get feature columns from the loaded models
clf_feature_columns = hybrid_reloaded["classifier"]["feature_columns"]
reg_feature_columns = hybrid_reloaded["regressor"]["feature_columns"]


def pic50_to_tier(pic50_value):
    if pic50_value >= ACTIVE_CUTOFF:
        return "Active"
    elif pic50_value >= INACTIVE_CUTOFF:
        return "Intermediate"
    return "Inactive"

# Reindex X_test for each model
X_test_for_classifier = X_test_raw.reindex(columns=clf_feature_columns, fill_value=0)
X_test_for_regressor = X_test_raw.reindex(columns=reg_feature_columns, fill_value=0)

clf_test_pred = clf_model.predict(X_test_for_classifier)
reg_test_pred = reg_model.predict(X_test_for_regressor)

classifier_predicted_tier = np.array([LABEL_TO_TIER[c] for c in clf_test_pred])
regressor_derived_tier = pd.Series(reg_test_pred).apply(pic50_to_tier).values
true_tier_test = np.array([LABEL_TO_TIER[c] for c in y_test_class])

agreement_rate = float(np.mean(classifier_predicted_tier == regressor_derived_tier))
consistency_df = pd.DataFrame({
    "canonical_smiles": [features["smiles"][i] for i in test_idx],
    "true_tier": true_tier_test,
    "classifier_predicted_tier": classifier_predicted_tier,
    "regressor_derived_tier": regressor_derived_tier,
    "models_agree": classifier_predicted_tier == regressor_derived_tier,
})

print(f"Classifier vs. regressor-derived tier agreement on the test set: {agreement_rate:.1%}")
consistency_df.head()


---
# PART 2 -- External Validation on BindingDB (Frozen Pipeline)



In [ ]:
import warnings
warnings.filterwarnings("ignore")

import time
import numpy as np
import pandas as pd
import requests
import joblib
import matplotlib.pyplot as plt
from tqdm import tqdm
import zipfile
import os

import rdkit
from rdkit import Chem, RDLogger, DataStructs
from rdkit.Chem import Descriptors, inchi, rdFingerprintGenerator, MACCSkeys
from rdkit.Chem.MolStandardize import rdMolStandardize
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit.Chem import FilterCatalog
from rdkit.Chem.FilterCatalog import FilterCatalogParams

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    matthews_corrcoef, roc_auc_score, average_precision_score,
    balanced_accuracy_score, confusion_matrix,
    r2_score, mean_squared_error, mean_absolute_error
)

RDLogger.DisableLog("rdApp.*")
RANDOM_STATE = 42
N_BOOTSTRAP = 1000

REQUEST_HEADERS = {"User-Agent": "AKT1-external-validation/1.0 (research use)"}
AKT1_UNIPROT = "P31749"
AKT1_GENEID = 207
BINDINGDB_AFFINITY_CUTOFF_NM = 1_000_000

# Structure standardisation tools (same as Notebook 0)
_largest_fragment_remover = rdMolStandardize.LargestFragmentChooser()
_uncharger = rdMolStandardize.Uncharger()
_pains_params = FilterCatalogParams()
_pains_params.AddCatalog(FilterCatalogParams.FilterCatalogs.PAINS_A)
_pains_params.AddCatalog(FilterCatalogParams.FilterCatalogs.PAINS_B)
_pains_params.AddCatalog(FilterCatalogParams.FilterCatalogs.PAINS_C)
_pains_catalog = FilterCatalog.FilterCatalog(_pains_params)

print(f"RDKit: {rdkit.__version__}")

#  Load trained models and reference data

In [ ]:
# Load the hybrid predictor bundle (contains classifier + regressor + metadata)
hybrid = joblib.load("hybrid_predictor.joblib")
clf_bundle = hybrid["classifier"]
reg_bundle = hybrid["regressor"]

# Unpack classifier metadata (used for tier assignment, AD thresholds, etc.)
clf_model = clf_bundle["model"]
clf_model_name = clf_bundle["model_name"]
clf_feature_columns = clf_bundle["feature_columns"]
clf_decision_bias = clf_bundle["decision_bias"]
clf_descriptor_medians = clf_bundle["descriptor_medians"]
clf_correlation_dropped_columns = clf_bundle["correlation_dropped_columns"]
TIER_TO_LABEL = clf_bundle["tier_to_label"]
LABEL_TO_TIER = clf_bundle["label_to_tier"]
CLASS_LABELS_ORDERED = clf_bundle["class_labels_ordered"]
TIER_NAMES_ORDERED = clf_bundle["tier_names_ordered"]
ACTIVE_PIC50_CUTOFF = clf_bundle["active_pic50_cutoff"]
INACTIVE_PIC50_CUTOFF = clf_bundle["inactive_pic50_cutoff"]
AD_TANIMOTO_THRESHOLD = clf_bundle["ad_tanimoto_threshold"]
AD_KNN_K = clf_bundle["ad_knn_k"]
train_ecfp4 = np.asarray(clf_bundle["train_ecfp4"])

# Unpack regressor metadata
reg_model = reg_bundle["model"]
reg_model_name = reg_bundle["model_name"]
reg_feature_columns = reg_bundle["feature_columns"]
reg_descriptor_medians = reg_bundle["descriptor_medians"]
reg_correlation_dropped_columns = reg_bundle["correlation_dropped_columns"]

print(f"Loaded hybrid predictor:")
print(f"  Classifier: {clf_model_name} | {len(clf_feature_columns)} feature columns")
print(f"  Regressor:  {reg_model_name} | {len(reg_feature_columns)} feature columns")
print(f"Tiers -- Active: pIC50 >= {ACTIVE_PIC50_CUTOFF} | Intermediate: {INACTIVE_PIC50_CUTOFF} <= pIC50 < {ACTIVE_PIC50_CUTOFF} | Inactive: pIC50 < {INACTIVE_PIC50_CUTOFF}")

# FIX: verify model.classes_ order matches CLASS_LABELS_ORDERED
if hasattr(clf_model, "classes_"):
    if list(clf_model.classes_) != list(CLASS_LABELS_ORDERED):
        print(f"WARNING: model.classes_ order {list(clf_model.classes_)} does not "
              f"match CLASS_LABELS_ORDERED {list(CLASS_LABELS_ORDERED)}. "
              f"decision_bias adjustment may be mislabeled.")
    else:
        print("Class order check passed: model.classes_ matches CLASS_LABELS_ORDERED.")

# Load reference features and splits (needed for deduplication, scaffold novelty, descriptor computation)
features = joblib.load("features.pkl")
splits = joblib.load("splits.pkl")
descriptor_names = features["descriptor_names"]
desc_func_by_name = dict(Descriptors._descList)
seen_idx = np.concatenate([splits["train_idx"], splits["val_idx"], splits["test_idx"]])
seen_inchikeys = set(features["inchikey"][seen_idx])
print(f"{len(seen_inchikeys)} unique compounds already seen in train/val/test (will be excluded)")


#  Load frozen blend weight & decision threshold


In [ ]:
if "blend_weight" not in clf_bundle or "decision_threshold" not in clf_bundle:
    raise KeyError(
        "hybrid_predictor.joblib has no frozen blend_weight/decision_threshold. Run the updated "
        "Notebook 4 (which computes these via development-set OOF predictions and locks them into "
        "the bundle) before evaluating external data -- do not guess a value here."
    )
w = clf_bundle["blend_weight"]
threshold = clf_bundle["decision_threshold"]
MODEL_LABEL = "Hybrid" if 0 < w < 1 else ("Classifier-only" if w == 1.0 else "Regressor-only")
print(f"Loaded frozen pipeline: {MODEL_LABEL} (w={w}), decision threshold={threshold}")
print("These were selected on development-set OOF predictions in Notebook 4 -- not re-derived here.")


Loaded frozen pipeline: Hybrid (w=0.85), decision threshold=0.375
These were selected on development-set OOF predictions in Notebook 4 -- not re-derived here.


#  Diagnostic: train-vs-validation fingerprint similarity



In [ ]:
features = joblib.load("features.pkl")
splits = joblib.load("splits.pkl")

val_idx = splits["val_idx"]
val_ecfp4_arr = np.asarray(features["ecfp4"])[val_idx].astype(np.int32)
train_only_ecfp4_arr = np.asarray(features["ecfp4"])[splits["train_idx"]].astype(np.int32)  # train_idx ONLY
val_pic50 = np.asarray(features["pIC50"])[val_idx]

inter = val_ecfp4_arr @ train_only_ecfp4_arr.T
pa = val_ecfp4_arr.sum(axis=1, keepdims=True)
pb = train_only_ecfp4_arr.sum(axis=1, keepdims=True).T
union = pa + pb - inter
tan = np.divide(inter, union, out=np.zeros_like(inter, dtype=float), where=union > 0)
val_nn_sim = tan.max(axis=1)

print(f"Median NN Tanimoto (validation vs. train-only): {np.median(val_nn_sim):.3f}")
for cutoff in [0.70, 0.85, 0.95]:
    print(f"  % >= {cutoff}: {(val_nn_sim >= cutoff).mean()*100:.1f}%")

print("\nIf validation performance concentrates in the high-similarity band (checked qualitatively "
      "\nabove; a formal per-band MCC breakdown can be added once frozen (w, threshold) predictions "
      "\nfor this validation set are available), that supports the fingerprint-similarity explanation "
      "\nfor any internal/external performance gap, rather than a leakage bug.")


#  Fetch AKT1 bioactivity data from BindingDB

In [ ]:
def _clean_key(k):
    """BindingDB sometimes prefixes fields with 'bdb.' (e.g. 'bdb.smile') depending on which endpoint/
    version answers the request. Normalize away the prefix so downstream code has one consistent schema."""
    return k.split(".", 1)[1] if k.startswith("bdb.") else k


def fetch_bindingdb_by_uniprot(uniprot_id, cutoff_nm=BINDINGDB_AFFINITY_CUTOFF_NM, max_retries=4):
    """Fetch all BindingDB binding data for one UniProt target. Returns a list of raw affinity
    records (dicts) with normalized (unprefixed) keys."""
    url = (
        "https://bindingdb.org/rest/getLigandsByUniprots"
        f"?uniprot={uniprot_id}&cutoff={cutoff_nm}&response=application/json"
    )
    last_exc = None
    data = None
    for attempt in range(max_retries):
        try:
            resp = requests.get(url, headers=REQUEST_HEADERS, timeout=180)
            resp.raise_for_status()
            text = resp.text.strip()
            if not text:
                return []  # BindingDB returns an empty body when nothing matches the UniProt ID
            data = resp.json()
            break
        except (requests.exceptions.RequestException, ValueError) as e:
            last_exc = e
            time.sleep(5 * (attempt + 1))
    else:
        raise RuntimeError(f"BindingDB did not return usable results for UniProt {uniprot_id} "
                            f"after {max_retries} retries: {last_exc}")

    if not data:
        return []
    payload = next(iter(data.values()), None)  # grab positionally -- see note above on the envelope key
    if not isinstance(payload, dict):
        return []
    raw_affinities = payload.get("affinities") or payload.get("bdb.affinities") or []
    return [{_clean_key(k): v for k, v in item.items()} for item in raw_affinities]


records = fetch_bindingdb_by_uniprot(AKT1_UNIPROT)
raw_bindingdb = pd.DataFrame(records)
if raw_bindingdb.empty:
    raw_bindingdb = pd.DataFrame(columns=["query", "monomerid", "smile", "affinity_type", "affinity", "pmid", "doi"])

print(f"Raw BindingDB records for UniProt {AKT1_UNIPROT}: {len(raw_bindingdb):,}")
if len(raw_bindingdb) < 20:
    print("WARNING: unexpectedly few records -- check network access and the UniProt ID before proceeding.")
raw_bindingdb.head()

#  Filter to quantitative potency records, compute pIC50

In [ ]:
QUANTITATIVE_ACTIVITY_NAMES = {"IC50", "KI", "KD", "EC50"}  # pooled as an approximate potency proxy --
                                                              # see Step 4 markdown / manuscript limitations note

bio_df = raw_bindingdb.copy()
print("Raw columns:", list(bio_df.columns))
n_raw_total = len(bio_df)
n_offtarget = 0

if len(bio_df) and "query" in bio_df.columns:
    print("\nTarget names in raw pull (sanity check before filtering):")
    print(bio_df["query"].value_counts())
    is_akt1 = bio_df["query"].astype(str).str.lower().str.contains("rac-alpha", na=False)
    n_offtarget = int((~is_akt1).sum())
    if n_offtarget:
        print(f"\nRemoving {n_offtarget} off-target record(s) not matching AKT1/RAC-alpha:")
        print(bio_df.loc[~is_akt1, "query"].value_counts())
    bio_df = bio_df[is_akt1].reset_index(drop=True)
    print(f"\nGenuine AKT1 records: {len(bio_df):,}")

bio_df["potency_type"] = bio_df["affinity_type"].astype(str).str.strip().str.upper()
bio_df = bio_df[bio_df["potency_type"].isin(QUANTITATIVE_ACTIVITY_NAMES)]
print(f"\nRows with a quantitative potency type (IC50/Ki/Kd/EC50): {len(bio_df):,}")


def parse_affinity(val):
    s = str(val).strip()
    if s == "" or s.lower() == "nan":
        return np.nan, None
    censor = None
    if s[0] in "<>":
        censor, s = s[0], s[1:].strip()
    try:
        return float(s), censor
    except ValueError:
        return np.nan, None


parsed = bio_df["affinity"].apply(parse_affinity)
bio_df["affinity_nM"] = parsed.apply(lambda t: t[0])
bio_df["affinity_censored"] = parsed.apply(lambda t: t[1])

n_censored = bio_df["affinity_censored"].notna().sum()
bio_df = bio_df[bio_df["affinity_censored"].isna()].dropna(subset=["affinity_nM"])
print(f"Dropped {n_censored} inequality-censored ('<' / '>') measurements without an exact value")

bio_df = bio_df[bio_df["affinity_nM"] > 0]  # guard against 0/negative before log10
bio_df = bio_df.dropna(subset=["smile"])
bio_df["pIC50"] = 9 - np.log10(bio_df["affinity_nM"])  # nM -> M -> -log10

print(f"\nQuantitative potency rows (IC50/Ki/Kd/EC50) with an exact value and SMILES: {len(bio_df):,} "
      f"across {bio_df['monomerid'].nunique():,} unique BindingDB compounds")
print("\nBreakdown by original measure:")
print(bio_df["potency_type"].value_counts())

attrition = [{"step": "1. Raw BindingDB records (UniProt " + AKT1_UNIPROT + ")", "n_rows": n_raw_total},
             {"step": "2. After AKT1/RAC-alpha target filter", "n_rows": n_raw_total - n_offtarget},
             {"step": "3. After IC50/Ki/Kd/EC50 + exact-value filter", "n_rows": len(bio_df)}]


#  Standardize structures & deduplicate by InChIKey (mirrors Notebook 0, Steps 7-8)

In [ ]:
def standardize_mol(smiles):
    mol = Chem.MolFromSmiles(str(smiles))
    if mol is None:
        return None
    try:
        mol = _largest_fragment_remover.choose(mol)
        mol = _uncharger.uncharge(mol)
        Chem.SanitizeMol(mol)
        return mol
    except Exception:
        return None


bio_df["mol"] = bio_df["smile"].apply(standardize_mol)
before = len(bio_df)
bio_df = bio_df.dropna(subset=["mol"]).reset_index(drop=True)
print(f"Rows with a valid, standardized structure: {len(bio_df):,} (dropped {before - len(bio_df):,} unparseable SMILES)")
attrition.append({"step": "4. After structure standardization", "n_rows": len(bio_df)})


def get_inchikey(mol):
    try:
        return inchi.MolToInchiKey(mol)
    except Exception:
        return None


bio_df["inchikey"] = bio_df["mol"].apply(get_inchikey)
before = len(bio_df)
bio_df = bio_df.dropna(subset=["inchikey"])

bio_df["pIC50"] = bio_df.groupby("inchikey")["pIC50"].transform("median")
# Record which potency measure(s) contributed to each compound's median pIC50 -- needed for the
# IC50/Ki/Kd/EC50 pooling limitation note in the manuscript.
bio_df["potency_type"] = bio_df.groupby("inchikey")["potency_type"].transform(
    lambda s: "+".join(sorted(s.unique()))
)
bio_df = bio_df.sort_values("monomerid").reset_index(drop=True)
bio_df = bio_df.drop_duplicates(subset="inchikey", keep="first").reset_index(drop=True)

print(f"After InChIKey dedup (median pIC50 across repeats): {len(bio_df):,} unique compounds "
      f"(dropped {before - len(bio_df):,} duplicate/repeat measurements)")
attrition.append({"step": "5. After InChIKey dedup", "n_rows": len(bio_df)})


#  Remove any compound already seen in training/validation/testing

In [ ]:
before = len(bio_df)
external_df = bio_df[~bio_df["inchikey"].isin(seen_inchikeys)].reset_index(drop=True)
print(f"External candidates before dedup-vs-training: {before:,}")
print(f"Removed as already seen in train/val/test: {before - len(external_df):,}")
print(f"Remaining, genuinely external: {len(external_df):,}")
attrition.append({"step": "6. After removing train/val/test overlap", "n_rows": len(external_df)})


#  PAINS / nuisance-compound filter

In [ ]:
def get_pains_alert(mol):
    entry = _pains_catalog.GetFirstMatch(mol)
    return entry.GetDescription() if entry is not None else None


external_df["pains_alert"] = external_df["mol"].apply(get_pains_alert)
n_flagged = external_df["pains_alert"].notna().sum()
before = len(external_df)
external_df = external_df[external_df["pains_alert"].isna()].drop(columns=["pains_alert"])
print(f"PAINS/nuisance alerts: {n_flagged} of {before} compounds flagged and removed")
attrition.append({"step": "7. After PAINS/nuisance filter", "n_rows": len(external_df)})


#  Lipinski Rule-of-Five filter

In [ ]:
external_df["MW"] = external_df["mol"].apply(Descriptors.MolWt)
external_df["LogP"] = external_df["mol"].apply(Descriptors.MolLogP)
external_df["HBD"] = external_df["mol"].apply(Descriptors.NumHDonors)
external_df["HBA"] = external_df["mol"].apply(Descriptors.NumHAcceptors)
external_df["TPSA"] = external_df["mol"].apply(Descriptors.TPSA)
external_df["RotBonds"] = external_df["mol"].apply(Descriptors.NumRotatableBonds)
external_df["HeavyAtoms"] = external_df["mol"].apply(Descriptors.HeavyAtomCount)

before = len(external_df)
external_df["ro5_violations"] = (
    (external_df["MW"] > 500).astype(int) + (external_df["LogP"] > 5).astype(int) +
    (external_df["HBD"] > 5).astype(int) + (external_df["HBA"] > 10).astype(int)
)
external_df = external_df[external_df["ro5_violations"] <= 1].reset_index(drop=True)
print(f"After Lipinski filter: {len(external_df):,} rows (dropped {before - len(external_df):,} violators)")
attrition.append({"step": "8. After Lipinski Ro5 filter", "n_rows": len(external_df)})


# Curation attrition summary (parity with Notebook 0's funnel chart)

In [ ]:
attrition_df = pd.DataFrame(attrition)
attrition_df["dropped"] = attrition_df["n_rows"].shift(1) - attrition_df["n_rows"]
attrition_df.to_csv("external_curation_attrition.csv", index=False)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh(attrition_df["step"][::-1], attrition_df["n_rows"][::-1], color="#4C78A8")
for i, v in enumerate(attrition_df["n_rows"][::-1]):
    ax.text(v, i, f"  {v:,}", va="center", fontsize=8)
ax.set_xlabel("Compounds remaining")
ax.set_title("External (BindingDB) curation attrition")
plt.tight_layout()
plt.savefig("figure_ext00_curation_attrition.png", dpi=300, bbox_inches="tight")
plt.show()
attrition_df

#  Assign tier labels

In [ ]:
def assign_bioactivity_tier(pic50):
    if pic50 >= ACTIVE_PIC50_CUTOFF:
        return "Active"
    elif pic50 >= INACTIVE_PIC50_CUTOFF:
        return "Intermediate"
    return "Inactive"

external_df["bioactivity_tier"] = external_df["pIC50"].apply(assign_bioactivity_tier)
external_df["y_true_label"] = external_df["bioactivity_tier"].map(TIER_TO_LABEL)
print("External set tier counts:")
print(external_df["bioactivity_tier"].value_counts().reindex(TIER_NAMES_ORDERED).fillna(0).astype(int))

External set tier counts:
bioactivity_tier
Inactive        108
Intermediate    101
Active           49
Name: count, dtype: int64


#  Compute molecular features (hybrid fingerprint + descriptors)

In [ ]:
_ecfp4_generator = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)

def morgan_fp(mol, generator=_ecfp4_generator, n_bits=2048):
    fp = generator.GetFingerprint(mol)
    arr = np.zeros((n_bits,), dtype=np.int8)
    DataStructs.ConvertToNumpyArray(fp, arr)
    return arr

def rdkit_topological_fp(mol, n_bits=2048):
    fp = Chem.RDKFingerprint(mol, fpSize=n_bits)
    arr = np.zeros((n_bits,), dtype=np.int8)
    DataStructs.ConvertToNumpyArray(fp, arr)
    return arr

def maccs_fp(mol):
    fp = MACCSkeys.GenMACCSKeys(mol)
    arr = np.zeros((167,), dtype=np.int8)
    DataStructs.ConvertToNumpyArray(fp, arr)
    return arr

def calc_2d_descriptors(mol):
    values = []
    for name in descriptor_names:
        func = desc_func_by_name.get(name)
        if func is None:
            values.append(np.nan)
            continue
        try:
            values.append(func(mol))
        except Exception:
            values.append(np.nan)
    return values

ext_mols = external_df["mol"].tolist()
ext_ecfp4 = np.array([morgan_fp(m) for m in tqdm(ext_mols, desc="ECFP4")])
ext_rdkit_topo = np.array([rdkit_topological_fp(m) for m in tqdm(ext_mols, desc="RDKit topological")])
ext_maccs = np.array([maccs_fp(m) for m in tqdm(ext_mols, desc="MACCS")])
ext_hybrid_fp = np.concatenate([ext_ecfp4, ext_rdkit_topo, ext_maccs], axis=1)
desc_results = [calc_2d_descriptors(m) for m in tqdm(ext_mols, desc="2D descriptors")]
ext_desc_all = np.array(desc_results)
desc_cols = features["desc_cols"]
ext_desc_df = pd.DataFrame(ext_desc_all, columns=desc_cols).replace([np.inf, -np.inf], np.nan)
fp_cols = [f"FP_{i}" for i in range(ext_hybrid_fp.shape[1])]
X_ext_all = pd.concat([
    pd.DataFrame(ext_hybrid_fp, columns=fp_cols),
    ext_desc_df.reset_index(drop=True)
], axis=1)
print(f"Raw external feature matrix: {X_ext_all.shape}")

# FIX 1: snapshot the pristine (un-imputed, no columns dropped) matrix here,
# before the classifier's preprocessing in Section 10 mutates X_ext_all.
# The regressor pipeline in Section 13 builds from this snapshot instead of
# from the classifier's already-processed X_ext_all.
X_ext_all_raw = X_ext_all.copy()


# Classifier feature matrix

In [ ]:
X_ext_clf = X_ext_all.copy()  # starts from the raw hybrid fingerprint + descriptors
X_ext_clf[desc_cols] = X_ext_clf[desc_cols].fillna(clf_descriptor_medians)
X_ext_clf = X_ext_clf.drop(columns=[c for c in clf_correlation_dropped_columns if c in X_ext_clf.columns])
missing_clf_cols = [c for c in clf_feature_columns if c not in X_ext_clf.columns]
if missing_clf_cols:
    raise ValueError(f"Missing classifier feature columns: {missing_clf_cols[:10]}")
X_ext_final = X_ext_clf[clf_feature_columns]
print(f"Final classifier feature matrix: {X_ext_final.shape}")


# Regressor feature matrix (1250 cols)

In [ ]:
X_ext_reg = X_ext_all_raw.copy()  # pristine, un-imputed, no columns dropped
X_ext_reg[desc_cols] = X_ext_reg[desc_cols].fillna(reg_descriptor_medians)
for col in reg_correlation_dropped_columns:
    if col in X_ext_reg.columns:
        X_ext_reg.drop(columns=[col], inplace=True)
missing_reg_cols = [c for c in reg_feature_columns if c not in X_ext_reg.columns]
if missing_reg_cols:
    raise ValueError(f"Missing regressor feature columns: {missing_reg_cols[:10]}")
X_ext_reg_final = X_ext_reg[reg_feature_columns]
print(f"Final regressor feature matrix: {X_ext_reg_final.shape}")

# Get classifier probabilities with decision_bias reweighting

In [ ]:
clf_proba = clf_model.predict_proba(X_ext_final)

if clf_decision_bias is not None:
    clf_proba_reweighted = clf_proba * np.exp(clf_decision_bias)
    # re-normalise to valid probability rows (defensive)
    clf_proba_reweighted = clf_proba_reweighted / clf_proba_reweighted.sum(axis=1, keepdims=True)
    clf_proba = clf_proba_reweighted
    print("Applied decision_bias reweighting to classifier probabilities.")


# Get regressor predictions and convert to one-hot probability rows

In [ ]:
pic50_pred = reg_model.predict(X_ext_reg_final)

n_samples = len(pic50_pred)
n_classes = len(CLASS_LABELS_ORDERED)  # should be 3: Inactive, Intermediate, Active
reg_proba = np.zeros((n_samples, n_classes))

for i, p in enumerate(pic50_pred):
    if p >= ACTIVE_PIC50_CUTOFF:
        reg_proba[i, TIER_TO_LABEL["Active"]] = 1.0
    elif p >= INACTIVE_PIC50_CUTOFF:
        reg_proba[i, TIER_TO_LABEL["Intermediate"]] = 1.0
    else:
        reg_proba[i, TIER_TO_LABEL["Inactive"]] = 1.0

print(f"Regressor one-hot probabilities shape: {reg_proba.shape}")

Regressor one-hot probabilities shape: (258, 3)


#  Blend classifier + regressor with the frozen (w, threshold) and compute metrics

In [ ]:
# w, threshold, and MODEL_LABEL were already loaded (frozen, from Notebook 4) in Section 2 --
# not re-derived here. This cell only applies them to the external set.
blended_proba = w * clf_proba + (1 - w) * reg_proba

# Binary active probability (class index for "Active")
active_col = TIER_TO_LABEL["Active"]
p_active = blended_proba[:, active_col]

# Ground truth binary labels
BINARY_ACTIVE_LABEL = TIER_TO_LABEL["Active"]
y_true_bin = (external_df["y_true_label"] == BINARY_ACTIVE_LABEL).astype(int).values

y_pred_bin = (p_active >= threshold).astype(int)


def compute_metrics(y_true, y_pred, y_score):
    """Accuracy/Precision/Recall/F1/MCC/BalancedAcc use thresholded predictions.
    ROC-AUC and PR-AUC use raw scores and are threshold-independent."""
    out = {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Balanced Accuracy": balanced_accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "MCC": matthews_corrcoef(y_true, y_pred),
    }
    if len(np.unique(y_true)) > 1:
        out["ROC-AUC"] = roc_auc_score(y_true, y_score)
        out["PR-AUC"] = average_precision_score(y_true, y_score)
    else:
        out["ROC-AUC"] = np.nan
        out["PR-AUC"] = np.nan
    return out


hybrid_metrics = compute_metrics(y_true_bin, y_pred_bin, p_active)

print(f"\nthreshold={threshold}: N={len(y_true_bin)}, N_active_pred={y_pred_bin.sum()}\n")
for name, val in hybrid_metrics.items():
    line = f"  {name:18s}: {val:.4f}" if not np.isnan(val) else f"  {name:18s}: n/a (single class)"
    print(line)

# Bootstrap CIs for every metric in the suite
rng = np.random.default_rng(RANDOM_STATE)
n = len(y_true_bin)
boot_rows = []
for _ in range(N_BOOTSTRAP):
    idx = rng.integers(0, n, n)
    yt, yp, ys = y_true_bin[idx], y_pred_bin[idx], p_active[idx]
    if len(np.unique(yt)) < 2:
        continue
    boot_rows.append(compute_metrics(yt, yp, ys))

boot_df = pd.DataFrame(boot_rows)
print(f"\nBootstrap ({len(boot_df)}/{N_BOOTSTRAP} resamples had both classes present):")
for col in boot_df.columns:
    vals = boot_df[col].dropna()
    lo, hi = np.percentile(vals, [2.5, 97.5])
    print(f"  {col:18s}: {vals.mean():.3f} (95% CI: {lo:.3f}-{hi:.3f})")


# Applicability domain (AD) and scaffold-novelty analysis

In [ ]:
A = ext_ecfp4.astype(np.int32)
B = train_ecfp4.astype(np.int32)

intersection = A @ B.T
popcount_A = A.sum(axis=1, keepdims=True)
popcount_B = B.sum(axis=1, keepdims=True).T
union = popcount_A + popcount_B - intersection
tanimoto = np.divide(intersection, union, out=np.zeros_like(intersection, dtype=float), where=union > 0)

nn_similarity = tanimoto.max(axis=1)
external_df["nn_tanimoto_to_train"] = nn_similarity
external_df["within_applicability_domain"] = nn_similarity >= AD_TANIMOTO_THRESHOLD

k = min(AD_KNN_K, tanimoto.shape[1])
knn_similarity = np.sort(tanimoto, axis=1)[:, -k:].mean(axis=1)
external_df["knn_mean_tanimoto_to_train"] = knn_similarity
external_df["within_applicability_domain_knn"] = knn_similarity >= AD_TANIMOTO_THRESHOLD

print(f"1-NN AD:        {external_df['within_applicability_domain'].mean()*100:.1f}% of external compounds "
      f"in-domain (NN Tanimoto >= {AD_TANIMOTO_THRESHOLD})")
print(f"kNN AD (k={k}): {external_df['within_applicability_domain_knn'].mean()*100:.1f}% of external compounds "
      f"in-domain (mean Tanimoto >= {AD_TANIMOTO_THRESHOLD})")

def get_murcko_scaffold(mol):
    try:
        scaf = MurckoScaffold.GetScaffoldForMol(mol)
        return Chem.MolToSmiles(scaf) if scaf is not None else None
    except Exception:
        return None

train_idx = splits["train_idx"]
if "smiles" not in features:
    raise KeyError(f"No 'smiles' key in features.pkl. Available keys: {list(features.keys())}. "
                    f"Update train_smiles below to the correct key.")
train_smiles = np.array(features["smiles"])[train_idx]

train_scaffolds = set()
for smi in tqdm(train_smiles, desc="Training scaffolds"):
    mol = Chem.MolFromSmiles(str(smi))
    if mol is None:
        continue
    scaf = get_murcko_scaffold(mol)
    if scaf:
        train_scaffolds.add(scaf)

external_df["murcko_scaffold"] = external_df["mol"].apply(get_murcko_scaffold)
external_df["scaffold_seen_in_training"] = external_df["murcko_scaffold"].isin(train_scaffolds)

print(f"\nScaffold novelty: {(~external_df['scaffold_seen_in_training']).mean()*100:.1f}% of external compounds "
      f"have a Murcko scaffold NOT seen in training ({len(train_scaffolds):,} unique training scaffolds)")

external_df["hybrid_correct"] = (y_pred_bin == y_true_bin)

print("\nAccuracy by 1-NN AD status:")
print(external_df.groupby("within_applicability_domain")["hybrid_correct"].agg(["mean", "count"]))
print("\nAccuracy by kNN AD status:")
print(external_df.groupby("within_applicability_domain_knn")["hybrid_correct"].agg(["mean", "count"]))
print("\nAccuracy by scaffold novelty:")
print(external_df.groupby("scaffold_seen_in_training")["hybrid_correct"].agg(["mean", "count"]))

def stratum_metrics(mask):
    yt, yp = y_true_bin[mask.values], y_pred_bin[mask.values]
    if len(np.unique(yt)) < 2:
        return {"n": int(mask.sum()), "balanced_acc": np.nan, "mcc": np.nan, "active_rate": yt.mean()}
    return {
        "n": int(mask.sum()),
        "balanced_acc": balanced_accuracy_score(yt, yp),
        "mcc": matthews_corrcoef(yt, yp),
        "active_rate": yt.mean(),
    }

for label, col in [("1-NN AD", "within_applicability_domain"),
                    ("kNN AD", "within_applicability_domain_knn"),
                    ("Scaffold seen in training", "scaffold_seen_in_training")]:
    print(f"\n{label} -- balanced accuracy / MCC (corrects for class-balance confound in raw accuracy):")
    for val in [False, True]:
        m = stratum_metrics(external_df[col] == val)
        print(f"  {col}={val}: n={m['n']:4d}  active_rate={m['active_rate']:.2f}  "
              f"balanced_acc={m['balanced_acc']:.3f}  MCC={m['mcc']:.3f}")



# Confusion matrix figure

In [ ]:
# --- FIX: this figure was referenced in the zip/export step but never generated -- added here. ---
from sklearn.metrics import ConfusionMatrixDisplay

fig, ax = plt.subplots(figsize=(5, 4.5))
cm = confusion_matrix(y_true_bin, y_pred_bin, labels=[0, 1])
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Not Active", "Active"])
disp.plot(ax=ax, cmap="Blues", colorbar=False, values_format="d")
ax.set_title(f"External validation (BindingDB) -- {MODEL_LABEL}\nthreshold={threshold}")
plt.tight_layout()
plt.savefig("figure_ext05_hybrid_confusion_matrix.png", dpi=300, bbox_inches="tight")
plt.show()


#  Save results

In [ ]:
if "test_metrics" in clf_bundle:
    comparison_hybrid = pd.DataFrame({
        "Internal test set": pd.Series(clf_bundle.get("test_metrics", {})),
        "External (BindingDB)": pd.Series(hybrid_metrics),
    })
    comparison_hybrid["Gap (internal - external)"] = comparison_hybrid["Internal test set"] - comparison_hybrid["External (BindingDB)"]
else:
    comparison_hybrid = pd.DataFrame({
        "External (BindingDB)": pd.Series(hybrid_metrics),
    })

external_validation_bundle = {
    "hybrid_model_name": f"Hybrid (clf={clf_model_name}, reg={reg_model_name}, w={w})",
    "classifier_model_name": clf_model_name,
    "regressor_model_name": reg_model_name,
    "n_external_compounds": len(external_df),
    "hybrid_metrics": hybrid_metrics,
    "internal_vs_external_hybrid": comparison_hybrid,
    "ad_accuracy_by_domain_1nn": external_df.groupby("within_applicability_domain")["hybrid_correct"].agg(["mean", "count"]),
    "ad_accuracy_by_domain_knn": external_df.groupby("within_applicability_domain_knn")["hybrid_correct"].agg(["mean", "count"]),
    "scaffold_novelty_pct": 100 * (~external_df["scaffold_seen_in_training"]).mean(),
    "source": f"BindingDB, target UniProt {AKT1_UNIPROT} (AKT1, human; NCBI Gene ID {AKT1_GENEID}; ChEMBL CHEMBL4282)",
}

joblib.dump(external_validation_bundle, "external_validation_results.joblib")
external_df.drop(columns=["mol"]).to_csv("external_validation_predictions.csv", index=False)

print("\nSaved external_validation_results.joblib and external_validation_predictions.csv")

#  Zip output files for download

In [ ]:
output_files = [
    "external_validation_results.joblib",
    "external_validation_predictions.csv",
    "figure_ext05_hybrid_confusion_matrix.png",
]
output_files = [f for f in output_files if os.path.exists(f)]
zip_path = "external_validation_outputs.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in output_files:
        zf.write(f)

print(f"\nZipped {len(output_files)} files into {zip_path}:")
for f in output_files:
    print(" -", f)

# Download zip (Colab)
try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    print("\nNot running in Colab - file saved locally at:", os.path.abspath(zip_path))